# Gold — Customer RFM Segmentation

Recency, frequency, and monetary analysis on **delivered** orders. Reference date = last order date in the dataset. Quintile scores map to named segments (Champions, Loyal, At Risk, Lost, …).

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.gold.customer_rfm as rfm_module
importlib.reload(rfm_module)

from src.gold.customer_rfm import (
    GoldRfmConfig,
    build_customer_rfm,
    segment_distribution,
    SEGMENT_THRESHOLDS,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = GoldRfmConfig()
print("Loaded from:", rfm_module.__file__)
print("Thresholds:", SEGMENT_THRESHOLDS)

In [ ]:
rfm, reference_date = build_customer_rfm(spark, config)
rfm.write.format("delta").mode("overwrite").saveAsTable(config.target_table)
print("Reference date:", reference_date)
print("Customers:", spark.table(config.target_table).count())
display(spark.table(config.target_table).orderBy("value_rank").limit(10))

In [ ]:
dist = segment_distribution(spark.table(config.target_table))
display(dist)

In [ ]:
import json

written = spark.table(config.target_table)
dist_rows = [row.asDict() for row in segment_distribution(written).collect()]

report = {
    "task": "gold_customer_rfm",
    "target_table": config.target_table,
    "reference_date": reference_date,
    "customer_count": written.count(),
    "segment_thresholds": SEGMENT_THRESHOLDS,
    "segment_distribution": dist_rows,
    "top_value_customers": [
        row.asDict() for row in written.orderBy("value_rank").limit(5).collect()
    ],
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/gold_customer_rfm.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)